In [3]:
import warnings
warnings.filterwarnings("ignore")
import os
import re
import numpy as np
import datetime
import psycopg2
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows',None)
from unidecode import unidecode
import networkx as nx
import re
hoy = datetime.date.today()
print(hoy)
fecha_inicio=datetime.date(2023,3,1)
fecha_trunc=(fecha_inicio).replace(day=1).strftime('%Y-%m-%d')
fecha_ini_pqr=datetime.date(2023,2,1)
fecha_fin=datetime.date(2023,5,23)

2024-06-19


### Preprocesamiento base detalle IVR

In [6]:
# Paso 1: Leer datos y preparar el DataFrame
df = pd.read_csv(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\vw_interaxa_detalle_ivr_202403121110.csv",sep=';')

In [7]:
base_principal=df.copy()
base_principal['ani'] = base_principal['ani'].astype(str)
base_principal['ani'] = base_principal['ani'].str.replace("^57", "", regex=True)
# base_principal.isnull().sum()
base_principal = base_principal.dropna(subset=["opcionesnavegaciontrazaopciones"])
base1=base_principal.copy()
# Separamos la columna opcionesnavegaciontrazaopciones por "|"
base_principal = base_principal.set_index('idtransaccion')['opcionesnavegaciontrazaopciones'].str.split('|', expand=True).stack().reset_index(level=1, drop=True).reset_index()
base_principal.columns = ['idtransaccion', 'opcionesnavegaciontrazaopciones']

df=base_principal.copy()
# Eliminamos filas con opcionesnavegaciontrazaopciones vacías
df = df[df['opcionesnavegaciontrazaopciones'].notna() & (df['opcionesnavegaciontrazaopciones'] != '')]

# Convertir la columna 'opcionesnavegaciontrazaopciones' a tipo string
df['opcionesnavegaciontrazaopciones'] = df['opcionesnavegaciontrazaopciones'].astype(str)

# Luego puedes continuar con tu código para dividir la cadena y realizar otras operaciones.
df[['op_num', 'op_text', 'op_tiempo']] = df['opcionesnavegaciontrazaopciones'].str.split(';', expand=True)

# Paso 2: Creamos columnas a partir de los valores que se encuentran separados por ";"
df[['op_num', 'op_text', 'op_tiempo']] = df['opcionesnavegaciontrazaopciones'].str.split(';', expand=True)

# Paso 3: Enumeramos las filas para cada idtransaccion, creando un bloque que marque paso por paso la navegación del IVR del cliente
df['bloque'] = df.groupby('idtransaccion').cumcount()


### Base de trazas

In [8]:
# Llamamos el archivo que contiene los tipos de traza del IVR
trazas=pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\TrazaOpcionesIVR Alkosto_tipos.xlsx",sheet_name='MaestroTrazasOpciones')
seg=trazas[['TrazaOpcion','Segmentacion']]
seg=seg.rename(columns={'TrazaOpcion':'op_text'})
#df.shape
#Eliminamos espacios que puedan estar al inicio o al final del los valores en la columna op_text
df['op_text']=df['op_text'].str.strip()
# Realizamos un merge entre el dataframe del paso anterior y este
df_join = pd.merge(df,seg, on=["op_text"], how='left')

# Marcar filas no identificadas, es decir, las que quedaron vacias como "no identificado"
df_join['Segmentacion'] = df_join['Segmentacion'].fillna('no identificado')

In [9]:
# Dentro del archivo de trazas no se encuentran los puntos en los que aparece el paso a asesor, así que lo crearemos manualmente como parte de la segmentación que queremos mantener
# df_join[df_join.Segmentacion=='no identificado']['op_num'].unique()
df_join.loc[df_join['op_text'].str.lower().str.contains('paso', case=False), 'Segmentacion'] = 'Paso asesor'

In [10]:
#Creamos una nueva columna la cual hace referencia a la siguiente marcación 
df['op_text_final']=df['op_text']

def transformar_dataframe(df):
    # Desplaza la columna "op_text" una posición hacia abajo dentro de cada grupo de idtransacción
    df['op_text_final'] = df.groupby("idtransaccion")["op_text"].shift(-1)

    # Identifica las últimas filas de cada grupo de idtransacción y las etiqueta como "final"
    last_rows = df.groupby("idtransaccion").tail(1).index
    df.loc[last_rows, 'op_text_final'] = 'final'

    return df


In [11]:
df_transformado = transformar_dataframe(df.copy())

# df.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_transacciones.xlsx",index=False)
# df.op_text.value_counts()
valores_unicos = {}

# Creamos las columnas NODO - ¡Este paso fue innecesario!
for i in range(1, 11):
    # Extraemos los valores únicos de la columna actual y eliminar los valores vacíos
    valores = trazas[f'Nodo {i}'].unique()
    valores_sin_vacios = [valor for valor in valores if valor != '']
    
    # Almacenamos los valores únicos en el diccionario
    valores_unicos[f'Nodo {i}'] = valores_sin_vacios

def obtener_nodo(op_tex):
    for nodo, valores in valores_unicos.items():
        if op_tex in valores:
            return nodo
    return None

df_transformado["Nodos"] = df_transformado["op_text"].apply(obtener_nodo)
df_transformado["Nodos"].value_counts()
df_transformado['Nodos'] = df_transformado['Nodos'].fillna('no identificado')


In [12]:
# Recontamos la columna bloque, ya que al haber eliminado aquellas trazas que no eran de navegación o resolutivas la cuenta se descuadro
df_transformado['bloque'] = df_transformado.groupby('idtransaccion').cumcount()


In [18]:
df_transformado.to_excel('df_transformado.xlsx',index=False)

df_transformado es el dataframe que se utiliza para crear los grafos

### Transformación final para la base extendida

In [19]:
df=df_transformado.copy()

# Agrupamos op_text y tiempo de duración para cada idtransaccion
ultima_op_tiempo = df.groupby('idtransaccion').agg({'op_text': 'last', 'op_tiempo': 'last'}).reset_index()
ultima_op_tiempo.rename(columns={'op_text': 'traza_final'}, inplace=True)
ultima_op_tiempo.rename(columns={'op_tiempo': 'tiempo_traza_final'}, inplace=True)

# Crear una columna para numerar las trazas para cada idtransaccion
df['num_traza'] = df.groupby('idtransaccion').cumcount() + 1

# Pivotaeamos el dataframe para tener las trazas como columnas
df_pivot = df.pivot(index='idtransaccion', columns='num_traza', values='op_text')

# Calculamos el total de opciones
df_pivot['total_opciones'] = df_pivot.count(axis=1)

# Renombramos las columnas de traza
df_pivot.columns = [f"traza_{col}" if col != 'total_opciones' else col for col in df_pivot.columns]

df_pivot.reset_index(inplace=True)

# última op_text y tiempo de duración
df_final = pd.merge(df_pivot, ultima_op_tiempo, on='idtransaccion')


In [20]:
# df = pd.read_csv(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\vw_interaxa_detalle_ivr_202403121110.csv", sep=";")
df=base1.copy()

In [21]:
df_coincidencias = pd.merge(left=df_final, right=df, on='idtransaccion', how='inner')


In [22]:
df=df_coincidencias.copy()

### Primer calculo recontactos

In [23]:
# Convertir las columnas de fechas a objetos datetime
df["fechahoraingreso"] = pd.to_datetime(df["fechahoraingreso"], format="%d/%m/%Y %H:%M")
df["fechahorafin"] = pd.to_datetime(df["fechahorafin"], format="%d/%m/%Y %H:%M")

df = df.sort_values(by=['ani', 'fechahoraingreso'])
df['recontactos'] = df.groupby('ani')['ani'].transform('size')
df['unico_contacto'] = df['recontactos'] <= 1
# Convertir la columna a tipo booleano
df['unico_contacto'] = df['unico_contacto'].astype('bool')

df['llamada'] = df.groupby('ani').cumcount()+1
# Calcular la duración del mensaje la cual no es el tiempo de duración del mensaje si no el tiempo transcurrido entre la primera llamada y la siguiente de cada usuario
df['duracion_mensaje'] =df['fechahoraingreso'] -df['fechahorafin'].shift(1)

# formato fecha
df['duracion_mensaje'] = pd.to_timedelta(df['duracion_mensaje'], errors='coerce')
 
# Definir las condiciones para agrupar los duracion_mensajes
conditions = [
    (df['duracion_mensaje'] >= pd.Timedelta(days=0)),
    (df['duracion_mensaje'] <= pd.Timedelta(days=1)),
    (df['duracion_mensaje'] == pd.Timedelta(days=2)),
    (df['duracion_mensaje'] == pd.Timedelta(days=3)),
    (df['duracion_mensaje'] == pd.Timedelta(days=4)),
    (df['duracion_mensaje'] == pd.Timedelta(days=5)),
    (df['duracion_mensaje'] == pd.Timedelta(days=10)),
    (df['duracion_mensaje'] == pd.Timedelta(days=15)),
    (df['duracion_mensaje'] >= pd.Timedelta(days=30)),
    (df['duracion_mensaje'].isna())  # Considerar los valores no válidos
]
 
# Definir las etiquetas para cada grupo
labels = ['a. Mismo día', 'b. 1 dias','c. 2 días', 'd. 3 días','e. 4 días', 'f. 5 días', 'g. 10 días', 'h. 15 días','i. 30 días']
 
# Definir los bordes de los bins
bins = [
    -pd.Timedelta(days=1), pd.Timedelta(days=1), pd.Timedelta(days=2), 
    pd.Timedelta(days=3), pd.Timedelta(days=4), pd.Timedelta(days=5), 
    pd.Timedelta(days=10), pd.Timedelta(days=15), pd.Timedelta(days=30),
    pd.Timedelta(days=365)
]
 
# etiquetamos para que nuestra columna grupo tenga los valores de la cantidad de días entre los contactos de los clientes 
df['grupo'] = pd.cut(df['duracion_mensaje'], bins=bins, labels=labels, include_lowest=True)

df['duracion_mensaje'] = df['duracion_mensaje'].astype(str)

# Modificar 'duracion_mensaje' solo en casos específicos
df['duracion_mensaje'] = np.where(df['llamada'] == 1, 'primer contacto', df['duracion_mensaje'])
df['duracion_mensaje'] = np.where(df['unico_contacto'] == True, 'unico contacto', df['duracion_mensaje'])

# luego de usar cut para etiquetar, las columnas quedan con un formato raro, a continuación transformamos a string para poder dejar marcados el primer y unico contacto
df['grupo'] = df['grupo'].map(str)

# Modificar 'grupo' solo en casos específicos
df['grupo'] = np.where(df['llamada'] == 1, 'primer contacto', df['grupo'])
df['grupo'] = np.where(df['unico_contacto'] == True, 'unico contacto', df['grupo'])

df['duracionivr_min']=df['duracionivrtotal']/60000


In [24]:
#Traemos una base con el tipo de desconexión para anexarlo
# desconexion=pd.read_csv(r"C:\Users\kmoralgu\Downloads\2024-04-09 Interacciones.csv", sep=';')
# desconexion=desconexion[['Tipo de desconexión','ID de conversación']]
# df_final=df.merge(desconexion, how='left',right_on='ID de conversación',left_on='idtransaccion')

In [25]:
# df_final['Tipo de desconexión'].isnull().sum()

In [26]:
# df_final.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final_sin_paso.xlsx",index=False)
# base=pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final.xlsx")

In [27]:
df.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final_inicio_ivr.xlsx",index=False)

In [39]:
df_transformado.head(3)

,idtransaccion,opcionesnavegaciontrazaopciones,op_num,op_text,op_tiempo,bloque,op_text_final,Nodos
1,b509e2c6-4983-45fe-bf85-23b27994aa4d,600;Inicio IVR inbound transfers;0,600,Inicio IVR inbound transfers,0,0,Oficina Alkosto a SAC,no identificado
2,b509e2c6-4983-45fe-bf85-23b27994aa4d,603;Oficina Alkosto a SAC;34,603,Oficina Alkosto a SAC,34,1,Paso agente SAC,no identificado
3,b509e2c6-4983-45fe-bf85-23b27994aa4d,208;Paso agente SAC;79,208,Paso agente SAC,79,2,Bienvenida encuesta SAC,no identificado


In [33]:
conditions = (
    ((df_transformado['op_text'] == 'Inicio IVR') & (df_transformado['op_text_final'] == 'final')) |
    ((df_transformado['op_text'] == 'Confirmar documento') & (df_transformado['op_text_final'] == 'final')) |
    ((df_transformado['op_text'] == 'Modificar documento') & (df_transformado['op_text_final'] == 'final')) |
    ((df_transformado['op_text'] == 'Menu principal') & (df_transformado['op_text_final'] == 'final'))
)

filtered_df_transformado = df_transformado.loc[conditions]

In [40]:
filtered_ids = filtered_df_transformado['idtransaccion']
df_transformado_filtrado = df_transformado[df_transformado['idtransaccion'].isin(filtered_ids)]

In [64]:
counts = df_transformado_filtrado.groupby('idtransaccion').count()['bloque']

condition = counts <= 4

valid_idtransaccion = condition[condition].index

df_final = df_transformado_filtrado[df_transformado_filtrado['idtransaccion'].isin(valid_idtransaccion)]

In [65]:
df_final.to_excel('base_menu.xlsx',index=False)

In [55]:
df_transformado_filtrado.to_excel(r'D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final_menu.xlsx',index=False)

In [51]:
df_transformado_filtrado.to_excel('base_final_menu.xlsx',index=False)

In [43]:
filtro=df_transformado_filtrado

In [46]:
lista=filtro.idtransaccion.tolist()

In [ ]:
# Ahora puedes aplicar el código que mencionaste
base = pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final_estado.xlsx")
df_garantias = df_transformado[df_transformado['idtransaccion'].isin(lista_garantias)]

df_garantias.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_entrega.xlsx",index=False)

### CREACIÓN DATAFRAME PARA GRAFOS

#### Garantia general

In [35]:
# Filtrar los idtransaccion que contienen la palabra "garantia" en df_transformado
filtro_garantia = df_transformado['opcionesnavegaciontrazaopciones'].str.contains('Estado de entrega', case=False)
# Crear una lista de los idtransaccion que cumplen con el filtro
lista_garantias = df_transformado.loc[filtro_garantia, 'idtransaccion'].tolist()
# Ahora puedes aplicar el código que mencionaste
base = pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final_estado.xlsx")
df_garantias = df_transformado[df_transformado['idtransaccion'].isin(lista_garantias)]

df_garantias.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_entrega.xlsx",index=False)

#### Menú principal

In [33]:
# base=pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final.xlsx")
# menu_ivr=base[base.traza_final=='Menu principal']
# lista_menu=menu_ivr.idtransaccion.tolist()
# df_menu = df_transformado[df_transformado['idtransaccion'].isin(lista_menu)]
# df_menu.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_menu.xlsx",index=False)

#### Transferencia alkosto-tuya

In [36]:
base=pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final.xlsx")
trans_ivr=base[base.traza_final=='Transferencia Alkosto - Tuya']
lista_trans=trans_ivr.idtransaccion.tolist()
df_trans = df_transformado[df_transformado['idtransaccion'].isin(lista_trans)]

In [37]:
df_trans5=df_trans.groupby('idtransaccion')['bloque'].max()
ani_hasta_4_veces = df_trans5[df_trans5 <= 4].index.tolist()
df_trans.loc[df_trans.idtransaccion.isin(ani_hasta_4_veces),'marca']="Sí"
df_trans.marca.fillna("No",inplace=True)

In [38]:
df_trans[df_trans.marca=='Sí'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_transferencia_hasta_4.xlsx",index=False)
df_trans[df_trans.marca=='No'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_transferencia_mas_4.xlsx",index=False)

#### Existencia de producto

In [39]:
#base=pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final.xlsx")
producto=base[base.traza_final=='Existencia de producto']
lista_producto=producto.idtransaccion.tolist()
df_producto= df_transformado[df_transformado['idtransaccion'].isin(lista_producto)]

In [40]:
df_producto5=df_producto.groupby('idtransaccion')['bloque'].max()
ani_hasta_4_veces = df_producto5[df_producto5 <= 5].index.tolist()
df_producto.loc[df_producto.idtransaccion.isin(ani_hasta_4_veces),'marca']="Sí"
df_producto.marca.fillna("No",inplace=True)

In [41]:
df_producto[df_producto.marca=='Sí'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_producto_hasta_5.xlsx",index=False)
df_producto[df_producto.marca=='No'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_producto_mas_5.xlsx",index=False)

#### Estado de entrega

In [42]:
#base=pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final.xlsx")
entrega=base[base.traza_final=='Estado de entrega']
lista_entrega=entrega.idtransaccion.tolist()
df_entrega= df_transformado[df_transformado['idtransaccion'].isin(lista_entrega)]
df_entrega5=df_entrega.groupby('idtransaccion')['bloque'].max()
ani_hasta_4_veces = df_entrega5[df_entrega5 <= 5].index.tolist()
df_entrega.loc[df_entrega.idtransaccion.isin(ani_hasta_4_veces),'marca']="Sí"
df_entrega.marca.fillna("No",inplace=True)
df_entrega[df_entrega.marca=='Sí'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_entrega_hasta_5.xlsx",index=False)
df_entrega[df_entrega.marca=='No'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_entrega_mas_5.xlsx",index=False)

In [45]:
# Filtrar los idtransaccion que contienen la palabra "garantia" en df_transformado
filtro_garantia = df_transformado['opcionesnavegaciontrazaopciones'].str.contains('Estado de entrega', case=False)
# Crear una lista de los idtransaccion que cumplen con el filtro
lista_garantias = df_transformado.loc[filtro_garantia, 'idtransaccion'].tolist()
# Ahora puedes aplicar el código que mencionaste
#base = pd.read_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_final.xlsx")
df_garantias = df_transformado[df_transformado['idtransaccion'].isin(lista_garantias)]

df_garantias.to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_grafos_estado_de_entrega.xlsx",index=False)

#### Garantias y devoluciones

In [48]:
garantia=base[base.traza_final=='Periodo igual o menor a 30 dias']
lista_garantia=garantia.idtransaccion.tolist()
df_garantia= df_transformado[df_transformado['idtransaccion'].isin(lista_garantia)]
df_garantia5=df_garantia.groupby('idtransaccion')['bloque'].max()
ani_hasta_4_veces = df_garantia5[df_garantia5 <= 6].index.tolist()
df_garantia.loc[df_garantia.idtransaccion.isin(ani_hasta_4_veces),'marca']="Sí"
df_garantia.marca.fillna("No",inplace=True)
df_garantia[df_garantia.marca=='Sí'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_garantia_menor_hasta_6.xlsx",index=False)
df_garantia[df_garantia.marca=='No'].to_excel(r"D:\OneDrive - Emtelco\Desarollo\Alkosto\base\base_garantia_menor_mas_6.xlsx",index=False)